# Analisis de menciones de tickers en Reddit (dumps .zst)

**Proyecto**: Tesis - Burbujas bursatiles y redes sociales | **Autor**: Pedro Piza

Este notebook usa las funciones del script `analisis_menciones_reddit.py`
(debe estar en la misma carpeta). Pasos:

1. Cargar el dump `*_submissions.zst` a un DataFrame de pandas
2. EDA: estructura y principales caracteristicas de la base
3. Cargar la Lista Maestra de Tickers y construir diccionarios de busqueda
4. Detectar menciones (symbol, $symbol, nombre de empresa)
5. Generar una matriz por ticker: renglones = fechas (diario), columnas = subreddits, valores = menciones

**Requisitos** (una sola vez): `pip install zstandard pandas openpyxl matplotlib`

In [4]:
# Paso 0 - imports
import pandas as pd
import matplotlib.pyplot as plt

import analisis_menciones_reddit as amr

amr.GRAFICAS = False        # en el notebook graficamos inline, no a PNG
amr.FILTRO_ANIO = 2025      # cambia a None para usar todos los anios (2013-2025)

print("Archivos .zst encontrados:", [a.name for a in amr.ARCHIVOS_ZST])
print("Filtro de anio:", amr.FILTRO_ANIO)

Archivos .zst encontrados: ['DaystromInstitute_submissions.zst']
Filtro de anio: 2025


## Paso 1 - Cargar el dump .zst a pandas

In [3]:
frames = []
for ruta in amr.ARCHIVOS_ZST:
    df = amr.leer_zst(ruta, filtro_anio=amr.FILTRO_ANIO)
    print(f"{ruta.name}: {len(df):,} posts cargados")
    frames.append(df)

posts = pd.concat(frames, ignore_index=True)
posts.head()

DaystromInstitute_submissions.zst: 813 posts cargados


,id,created_utc,subreddit,author,title,selftext,score,num_comments,upvote_ratio,permalink,fecha_hora,fecha,texto_completo
0,1hqzua7,1.735725e+09,DaystromInstitute,Agreeable-Divide-150,"Knowing what we know about Captain Picard, sho...",[removed],1,12,0.26,/r/DaystromInstitute/comments/1hqzua7/knowing_...,2025-01-01 09:42:28+00:00,2025-01-01,"Knowing what we know about Captain Picard, sho..."
1,1hrk882,1.735788e+09,DaystromInstitute,LunchyPete,"In 'Yesteryear', was one Spock's consciousness...","In the episode 'Yesteryear', when Spock realiz...",11,11,0.76,/r/DaystromInstitute/comments/1hrk882/in_yeste...,2025-01-02 03:14:06+00:00,2025-01-02,"In 'Yesteryear', was one Spock's consciousness..."
2,1hrpbyg,1.735807e+09,DaystromInstitute,treefox,"Discovery should have started with ""Context is...","On a whim, I rewatched ""Context is for Kings"" ...",158,18,0.93,/r/DaystromInstitute/comments/1hrpbyg/discover...,2025-01-02 08:35:55+00:00,2025-01-02,"Discovery should have started with ""Context is..."
3,1hrx632,1.735834e+09,DaystromInstitute,eliwood98,Is Vic Fontaine designed as a therapy program?,"Ok, Vic Fontaine is an absolutely fantastic ch...",74,46,0.91,/r/DaystromInstitute/comments/1hrx632/is_vic_f...,2025-01-02 16:09:17+00:00,2025-01-02,Is Vic Fontaine designed as a therapy program?...
4,1hsgv1z,1.735889e+09,DaystromInstitute,wtffu006,That fancy armour and transphasic torpedos fut...,[removed],1,2,1.00,/r/DaystromInstitute/comments/1hsgv1z/that_fan...,2025-01-03 07:16:43+00:00,2025-01-03,That fancy armour and transphasic torpedos fut...


## Paso 2 - EDA: principales caracteristicas de la base

In [ ]:
amr.eda(posts, "DaystromInstitute")

In [ ]:
# Graficas inline (paleta verde ITAM)
verde1, verde2 = "#00684A", "#1A5742"
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

por_mes = posts.set_index("fecha_hora").resample("ME").size()
axes[0].plot(por_mes.index, por_mes.values, color=verde1)
axes[0].set_title("Posts por mes", color=verde1)
axes[0].set_ylabel("posts")

posts["score"].clip(upper=posts["score"].quantile(0.99)).hist(
    bins=50, ax=axes[1], color=verde2)
axes[1].set_title("Distribucion de score (p99)", color=verde1)
plt.tight_layout()
plt.show()

In [ ]:
# Manipulacion libre del DataFrame - ejemplos
# posts mas comentados
posts.nlargest(5, "num_comments")[["fecha", "title", "score", "num_comments"]]

## Paso 3 - Lista Maestra de Tickers y diccionarios de busqueda

Reglas de deteccion (limitaciones documentadas en el .py):
- `symbol` como palabra en MAYUSCULAS (longitud >= 3, excluyendo acronimos comunes)
- `$symbol` como cashtag (cualquier longitud)
- `security_name` limpio ("Apple Inc. - Common Stock" -> "apple"), case-insensitive;
  se descartan nombres que son palabras comunes del ingles (Star, News, Strategy...)

In [ ]:
simbolos_token, simbolos_cash, nombres, lista_tickers = \
    amr.construir_diccionarios(amr.ARCHIVO_TICKERS)

lista_tickers.head()

In [ ]:
# prueba rapida del detector con un texto de ejemplo
ejemplo = "I bought $AAPL and MSFT yesterday. Apple and Nvidia keep going up."
amr.detectar_menciones(ejemplo, simbolos_token, simbolos_cash, nombres)

## Paso 4 - Deteccion de menciones en todos los posts

In [ ]:
largo = amr.procesar(posts, simbolos_token, simbolos_cash, nombres)
print(f"Registros (ticker, fecha, subreddit): {len(largo):,}")
largo.head(10)

## Paso 5 - Matrices por ticker

Una matriz por ticker: renglones = fechas diarias del periodo (dias sin
menciones = 0), columnas = subreddits. Se exporta un CSV por ticker a la
carpeta `matrices_menciones/`.

In [ ]:
matrices = amr.generar_matrices(largo, posts["fecha"].min(), posts["fecha"].max())

resumen = (largo.groupby("ticker")["menciones"].sum()
                .sort_values(ascending=False).reset_index())
resumen.to_csv(amr.CARPETA_SALIDA / "resumen_menciones_por_ticker.csv", index=False)

print(f"Tickers con al menos 1 mencion: {len(matrices):,}")
print(f"CSVs guardados en: {amr.CARPETA_SALIDA}")
resumen.head(20)

In [ ]:
# acceso a la matriz de un ticker especifico
ticker = "AAPL"          # cambia el ticker aqui
m = matrices[ticker]
print(f"Matriz {ticker}: {m.shape[0]} dias x {m.shape[1]} subreddit(s)")
m[m.sum(axis=1) > 0]     # solo dias con menciones

In [ ]:
# serie de tiempo de menciones de un ticker
m.plot(figsize=(12, 3), color="#00684A", title=f"Menciones diarias de {ticker}")
plt.ylabel("menciones")
plt.show()

---
**Notas**:
- El dump cubre 2013-2025 aunque el nombre sugiera solo 2025; usa `amr.FILTRO_ANIO`.
- En este subreddit (Star Trek) las menciones detectadas son mayormente ruido
  residual (ej. DSC = Star Trek Discovery); sirve como control negativo.
- Para procesar mas subreddits, copia sus `*_submissions.zst` a esta carpeta
  y vuelve a correr desde el Paso 1. Las matrices tendran una columna por subreddit.